## Import Dependencies

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, precision_recall_fscore_support

In [3]:
# Paths

root = "/kaggle/input/breast-cancer-detection-mri/Breast_Cancer_MRI_Dataset"
IMG_SIZE = (512, 512)
BATCH = 16
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

In [4]:
# Load dataset

preprocess_fn = tf.keras.applications.mobilenet_v2.preprocess_input

def build_ds(subdir, augment=False, shuffle=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        os.path.join(root, subdir),
        labels="inferred",
        label_mode="int",
        color_mode="rgb",
        batch_size=BATCH,
        image_size=IMG_SIZE,
        shuffle=shuffle,
        seed=SEED
    )

    aug = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.05),
        layers.RandomContrast(0.1),
        layers.RandomZoom(0.1)
    ]) if augment else None

    def _map(x, y):
        x = tf.cast(x, tf.float32)
        if aug is not None:
            x = aug(x, training=True)
        x = preprocess_fn(x)
        return x, y

    return (ds.map(_map, num_parallel_calls=AUTOTUNE)
              .cache()
              .prefetch(AUTOTUNE))

train_ds = build_ds("train", augment=True, shuffle=True)
val_ds   = build_ds("validation", augment=False, shuffle=False)
test_ds  = build_ds("test", augment=False, shuffle=False)

Found 4000 files belonging to 2 classes.


I0000 00:00:1761984945.081256      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1761984945.082111      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 400 files belonging to 2 classes.
Found 400 files belonging to 2 classes.


In [5]:
# Build model

base = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)

base.trainable = False  # Stage 1: freeze backbone

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid", kernel_regularizer=regularizers.l2(0.001))(x)
model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

/tmp/ipykernel_37/91125679.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = tf.keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [6]:
# Train head

model_path = "/kaggle/working/MOBILENETV2.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(model_path, monitor="val_auc", mode="max",
                                    save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_auc", patience=10, mode="max", restore_best_weights=True)
]

In [7]:
history1 = model.fit(train_ds, validation_data=val_ds, epochs=100, callbacks=callbacks, verbose=1)

Epoch 1/100


I0000 00:00:1761984959.678039     108 service.cc:148] XLA service 0x7a5018002f80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1761984959.679972     108 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1761984959.679990     108 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1761984961.408112     108 cuda_dnn.cc:529] Loaded cuDNN version 90300
E0000 00:00:1761984963.992985     108 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1761984964.247736     108 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


  2/250 ━━━━━━━━━━━━━━━━━━━━ 22s 90ms/step - accuracy: 0.6875 - auc: 0.5139 - loss: 0.6967   

I0000 00:00:1761984969.015858     108 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 472ms/step - accuracy: 0.5135 - auc: 0.5318 - loss: 0.7403
Epoch 1: val_auc improved from -inf to 0.72649, saving model to /kaggle/working/MOBILENETV2.keras
250/250 ━━━━━━━━━━━━━━━━━━━━ 143s 500ms/step - accuracy: 0.5136 - auc: 0.5318 - loss: 0.7403 - val_accuracy: 0.6750 - val_auc: 0.7265 - val_loss: 0.6519
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.5731 - auc: 0.5967 - loss: 0.6999
Epoch 2: val_auc improved from 0.72649 to 0.79997, saving model to /kaggle/working/MOBILENETV2.keras
250/250 ━━━━━━━━━━━━━━━━━━━━ 19s 75ms/step - accuracy: 0.5731 - auc: 0.5967 - loss: 0.6999 - val_accuracy: 0.7325 - val_auc: 0.8000 - val_loss: 0.6189
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.5850 - auc: 0.6141 - loss: 0.6943
Epoch 3: val_auc improved from 0.79997 to 0.83851, saving model to /kaggle/working/MOBILENETV2.keras
250/250 ━━━━━━━━━━━━━━━━━━━━ 19s 75ms/step - accuracy: 0.5850 - auc: 0.6141 - loss: 0.6943 - val_accu

In [8]:
# Evaluate with best threshold

model.load_weights("/kaggle/working/MOBILENETV2.keras")

def best_threshold(ds):
    y_true = np.concatenate([y for _, y in ds], axis=0)
    y_prob = model.predict(ds, verbose=0).ravel()
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    youden = tpr - fpr
    return thr[np.argmax(youden)]

thr = best_threshold(val_ds)
print("Best threshold:", thr)

def metrics_at_threshold(ds, thr):
    y_true = np.concatenate([y for _, y in ds], axis=0)
    y_prob = model.predict(ds, verbose=0).ravel()
    y_pred = (y_prob >= thr).astype(int)
    auc = roc_auc_score(y_true, y_prob)
    cm = confusion_matrix(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp)
    return dict(auc=auc, precision=prec, recall=rec, f1=f1, specificity=spec, cm=cm)

print("Validation metrics:", metrics_at_threshold(val_ds, thr))
print("Test metrics:", metrics_at_threshold(test_ds, thr))

Best threshold: 0.47766182
Validation metrics: {'auc': 0.9583499999999999, 'precision': 0.8970588235294118, 'recall': 0.915, 'f1': 0.9059405940594061, 'specificity': 0.895, 'cm': array([[179,  21],
       [ 17, 183]])}
Test metrics: {'auc': 0.91565, 'precision': 0.7818181818181819, 'recall': 0.86, 'f1': 0.819047619047619, 'specificity': 0.76, 'cm': array([[152,  48],
       [ 28, 172]])}


## PTQ and Fine-tuning


In [9]:
model.load_weights("/kaggle/working/MOBILENETV2.keras")
base.trainable = True # Unfreeze the backbone

In [21]:
callbacks_for_fine_tune = [
    keras.callbacks.ModelCheckpoint(model_path, monitor="val_auc", mode="max",
                                    save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_auc", patience=15, mode="max", restore_best_weights=True)
]

In [22]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-5), 
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

In [23]:
history2 = model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks, verbose=1)

Epoch 1/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 247ms/step - accuracy: 0.7747 - auc: 0.8585 - loss: 0.4754
Epoch 1: val_auc did not improve from 0.96169
250/250 ━━━━━━━━━━━━━━━━━━━━ 119s 268ms/step - accuracy: 0.7747 - auc: 0.8585 - loss: 0.4753 - val_accuracy: 0.7625 - val_auc: 0.9506 - val_loss: 0.4822
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 243ms/step - accuracy: 0.8545 - auc: 0.9309 - loss: 0.3417
Epoch 2: val_auc did not improve from 0.96169
250/250 ━━━━━━━━━━━━━━━━━━━━ 62s 249ms/step - accuracy: 0.8545 - auc: 0.9309 - loss: 0.3417 - val_accuracy: 0.7475 - val_auc: 0.9402 - val_loss: 0.5156
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 243ms/step - accuracy: 0.8925 - auc: 0.9545 - loss: 0.2790
Epoch 3: val_auc did not improve from 0.96169
250/250 ━━━━━━━━━━━━━━━━━━━━ 62s 248ms/step - accuracy: 0.8925 - auc: 0.9545 - loss: 0.2790 - val_accuracy: 0.7850 - val_auc: 0.9277 - val_loss: 0.4447
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step - accuracy: 0.9208 - auc: 0.9756 - loss: 0.20

KeyboardInterrupt: 

In [24]:
# Evaluate after fine tuning

model.load_weights("/kaggle/working/MOBILENETV2.keras")

def best_threshold(ds):
    y_true = np.concatenate([y for _, y in ds], axis=0)
    y_prob = model.predict(ds, verbose=0).ravel()
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    youden = tpr - fpr
    return thr[np.argmax(youden)]

thr = best_threshold(val_ds)
print("Best threshold:", thr)

def metrics_at_threshold(ds, thr):
    y_true = np.concatenate([y for _, y in ds], axis=0)
    y_prob = model.predict(ds, verbose=0).ravel()
    y_pred = (y_prob >= thr).astype(int)
    auc = roc_auc_score(y_true, y_prob)
    cm = confusion_matrix(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp)
    return dict(auc=auc, precision=prec, recall=rec, f1=f1, specificity=spec, cm=cm)

print("Validation metrics:", metrics_at_threshold(val_ds, thr))
print("Test metrics:", metrics_at_threshold(test_ds, thr))

Best threshold: 0.16740167
Validation metrics: {'auc': 0.9914749999999999, 'precision': 0.9747474747474747, 'recall': 0.965, 'f1': 0.9698492462311558, 'specificity': 0.975, 'cm': array([[195,   5],
       [  7, 193]])}
Test metrics: {'auc': 0.977175, 'precision': 0.9261083743842364, 'recall': 0.94, 'f1': 0.9330024813895781, 'specificity': 0.925, 'cm': array([[185,  15],
       [ 12, 188]])}


In [25]:
# !pip install tensorflow_model_optimization

In [26]:
import tensorflow_model_optimization as tfmot

In [27]:
quantize_model = tfmot.quantization.keras.quantize_model

In [28]:
# Load the weights
model.load_weights("/kaggle/working/MOBILENETV2.keras")

In [29]:
# Define a generator function for the representative dataset
def representative_data_gen():
    for input_value, _ in test_ds.take(100): 
        yield [input_value] # The converter expects a list of arrays

In [30]:
import tensorflow as tf
# Create the TFLite converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.representative_dataset = representative_data_gen

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Set the input and output tensors to uint8 
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

In [31]:
# Convert the model
tflite_quant_model = converter.convert()

Saved artifact at '/tmp/tmpqcqcbrqg'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 512, 512, 3), dtype=tf.float32, name='keras_tensor_159')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134487175397648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134487175399760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134487175398608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134487175399184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134487175398416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134487175399376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134487175402064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134487175400720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134487175401104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134487175398224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1344871754

/usr/local/lib/python3.11/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1761989766.843893      37 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1761989766.843927      37 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8


In [32]:
# Save the quantized model
quantized_model_path = "/kaggle/working/MOBILENETV2_8-BIT_PTQ.tflite"
with open(quantized_model_path, "wb") as f:
    f.write(tflite_quant_model)

In [33]:
files = [
    "/kaggle/working/MOBILENETV2.keras",
    "/kaggle/working/MOBILENETV2_8-BIT_PTQ.tflite"
]

for f in files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"{f}: {size_mb:.2f} MB")

/kaggle/working/MOBILENETV2.keras: 26.27 MB
/kaggle/working/MOBILENETV2_8-BIT_PTQ.tflite: 2.59 MB


In [34]:
interpreter = tf.lite.Interpreter(model_content=tflite_quant_model)
input_type = interpreter.get_input_details()[0]['dtype']
print('input: ', input_type)
output_type = interpreter.get_output_details()[0]['dtype']
print('output: ', output_type)

input:  <class 'numpy.uint8'>
output:  <class 'numpy.uint8'>
